In [0]:
%python
%pip install "h3>=3.7,<4.0" pygeohash


In [0]:
dbutils.library.restartPython()

In [0]:
from pyspark.sql.functions import (
    col, lit, upper, trim, sha2, concat,
    when, coalesce, current_date, row_number,
    regexp_replace, radians, sin, cos, sqrt,
    atan2, udf, explode, array
)
from pyspark.sql.types import StringType, ArrayType, DoubleType
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import math

#  UDFs 
@udf(StringType())
def to_h3(lat, lon):
    try:
        import h3
        return h3.geo_to_h3(float(lat), float(lon), resolution=9)
    except:
        return None


@udf(ArrayType(StringType()))
def h3_kring(h3_index, k):
    # Returns all H3 cells within k rings of the given cell
    # k=1 → ~300m, k=2 → ~500m, k=3 → ~800m, k=4 → ~1km
    try:
        import h3
        return list(h3.k_ring(h3_index, int(k)))
    except:
        return []

@udf(DoubleType())
def haversine_meters(lat1, lon1, lat2, lon2):
    # Exact distance in meters between two lat/lon points
    try:
        R = 6_371_000  # Earth radius in meters
        φ1, φ2 = math.radians(float(lat1)), math.radians(float(lat2))
        Δφ = math.radians(float(lat2) - float(lat1))
        Δλ = math.radians(float(lon2) - float(lon1))
        a = math.sin(Δφ/2)**2 + math.cos(φ1) * math.cos(φ2) * math.sin(Δλ/2)**2
        return round(R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a)), 2)
    except:
        return None

# Max search radius ─
# k=4 rings at resolution 9 ≈ 1km — covers all queries up to 1km
K_RING = 4
MAX_DISTANCE_M = 1000

In [0]:
hotel_raw = spark.table("cre_catalog.bronze.hotel_properties_raw")
rest_raw = spark.table("cre_catalog.bronze.restaurant_inspections_raw")

In [0]:
# Deduplicate to one row per PARID — keep latest tax year
w_hotel = Window.partitionBy("PARID").orderBy(col("TAXYEAR").desc())

dim_hotel_new = (
    hotel_raw
    .filter(
        col("Latitude").isNotNull() &
        col("Longitude").isNotNull()
    )
    .withColumn("rn", row_number().over(w_hotel))
    .filter(col("rn") == 1)
    .drop("rn")
    .withColumn("hotel_id", sha2(col("PARID").cast("string"), 256))
    .withColumn("h3_index", to_h3(col("Latitude").cast("string"), col("Longitude").cast("string")))
    .select(
        col("hotel_id"),                                    # PK
        col("PARID").cast("string").alias("parid"),
        col("OWNER_NAME").alias("owner_name"),
        col("BLDG_CLASS").alias("bldg_class"),
        col("STREET_NUMBER").cast("string").alias("street_number"),
        col("STREET_NAME").alias("street_name"),
        col("Postcode").cast("string").alias("zip_code"),
        col("Borough").alias("borough"),
        col("Latitude").cast("double").alias("latitude"),
        col("Longitude").cast("double").alias("longitude"),
        col("h3_index"),
        col("NTA_NAME").alias("nta_name"),
        current_date().alias("load_date")
    )
)

In [0]:
#  MERGE: insert new hotels, update if owner/location changed 
if spark.catalog.tableExists("cre_catalog.silver.dim_hotel"):
    dt = DeltaTable.forName(spark, "cre_catalog.silver.dim_hotel")
    dt.alias("tgt").merge(
        dim_hotel_new.alias("src"),
        "tgt.hotel_id = src.hotel_id"
    ).whenMatchedUpdate(condition="""
        tgt.owner_name  != src.owner_name  OR
        tgt.latitude    != src.latitude    OR
        tgt.longitude   != src.longitude
    """, set={
        "owner_name":    "src.owner_name",
        "latitude":      "src.latitude",
        "longitude":     "src.longitude",
        "h3_index":      "src.h3_index",
        "geohash":       "src.geohash",
        "load_date":     "src.load_date"
    }).whenNotMatchedInsertAll().execute()
    print(f"dim_hotel merged")
else:
    dim_hotel_new.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("cre_catalog.silver.dim_hotel")
    print(f"dim_hotel created")

print(f"   Total rows: {spark.table('cre_catalog.silver.dim_hotel').count():,}")

In [0]:

# Deduplicate to one row per restaurant (CAMIS) 
# Keep latest inspection by INSPECTION DATE
# Exclude rows with no lat/lon or inspection date = 1900-01-01 (never inspected)
w_rest = Window.partitionBy("CAMIS").orderBy(col("INSPECTION DATE").desc())

dim_restaurant_new = (
    rest_raw
    .filter(
        col("CAMIS").isNotNull() &
        col("Latitude").isNotNull() &
        col("Longitude").isNotNull() &
        (col("`INSPECTION DATE`") != "1900-01-01")
    )
    .withColumn("rn", row_number().over(w_rest))
    .filter(col("rn") == 1)
    .drop("rn")
    .withColumn(
        "restaurant_id",
        sha2(col("CAMIS").cast("string"), 256)
    )
    .withColumn("h3_index",  to_h3(col("Latitude").cast("string"), col("Longitude").cast("string")))
    .select(
        col("restaurant_id"),
        col("CAMIS").cast("string").alias("camis"),
        col("DBA").alias("name"),
        col("BORO").alias("borough"),
        col("BUILDING").alias("street_number"),
        col("STREET").alias("street_name"),
        col("ZIPCODE").cast("string").alias("zip_code"),
        col("`CUISINE DESCRIPTION`").alias("cuisine_type"),
        col("SCORE").cast("int").alias("inspection_score"),
        col("GRADE").alias("grade"),
        col("`GRADE DATE`").alias("grade_date"),
        col("`INSPECTION DATE`").alias("last_inspection_date"),
        col("Latitude").cast("double").alias("latitude"),
        col("Longitude").cast("double").alias("longitude"),
        col("h3_index"),
        col("NTA").alias("nta_code"),
        current_date().alias("load_date")
    )
)

In [0]:

#  MERGE: insert new restaurants, update if grade/score/location changed 
if spark.catalog.tableExists("cre_catalog.silver.dim_restaurant"):
    dt = DeltaTable.forName(spark, "cre_catalog.silver.dim_restaurant")
    dt.alias("tgt").merge(
        dim_restaurant_new.alias("src"),
        "tgt.restaurant_id = src.restaurant_id"
    ).whenMatchedUpdate(condition="""
        tgt.grade             != src.grade            OR
        tgt.inspection_score  != src.inspection_score OR
        tgt.latitude          != src.latitude         OR
        tgt.longitude         != src.longitude
    """, set={
        "grade":              "src.grade",
        "inspection_score":   "src.inspection_score",
        "grade_date":         "src.grade_date",
        "last_inspection_date": "src.last_inspection_date",
        "latitude":           "src.latitude",
        "longitude":          "src.longitude",
        "h3_index":           "src.h3_index",
        "geohash":            "src.geohash",
        "load_date":          "src.load_date"
    }).whenNotMatchedInsertAll().execute()
    print(f"dim_restaurant merged")
else:
    dim_restaurant_new.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("cre_catalog.silver.dim_restaurant")
    print(f"dim_restaurant created")

print(f"   Total rows: {spark.table('cre_catalog.silver.dim_restaurant').count():,}")

In [0]:
#  Load sites with H3 index 
sites = spark.table("cre_catalog.silver.dim_site") \
    .filter(
        col("is_current") == True  &
        col("h3_index").isNotNull() &
        col("latitude").isNotNull() &
        col("longitude").isNotNull()
    ) \
    .select("site_id", "latitude", "longitude", "h3_index")

#  Build unified amenity table 
hotels = spark.table("cre_catalog.silver.dim_hotel") \
    .filter(col("h3_index").isNotNull()) \
    .select(
        col("hotel_id").alias("amenity_id"),
        lit("hotel").alias("amenity_type"),
        col("latitude"), col("longitude"), col("h3_index")
    )

restaurants = spark.table("cre_catalog.silver.dim_restaurant") \
    .filter(col("h3_index").isNotNull()) \
    .select(
        col("restaurant_id").alias("amenity_id"),
        lit("restaurant").alias("amenity_type"),
        col("latitude"), col("longitude"), col("h3_index")
    )

amenities = hotels.union(restaurants)

In [0]:
# Step 1: Expand each site to all H3 cells within k rings 
# This dramatically reduces candidate pairs before computing Haversine
sites_expanded = (
    sites
    .withColumn(
        "neighbor_cells",
        h3_kring(col("h3_index"), lit(K_RING))
    )
    .withColumn("neighbor_cell", explode(col("neighbor_cells")))
    .drop("neighbor_cells")
)


#  Step 2: Join amenities on H3 cell match 
amenities_renamed = amenities.withColumnRenamed("h3_index", "amenity_h3")

candidates = sites_expanded.join(
    amenities_renamed,
    sites_expanded["neighbor_cell"] == amenities_renamed["amenity_h3"],
    "inner"
).select(
    col("site_id"),
    col("amenity_id"),
    col("amenity_type"),
    sites_expanded["latitude"].alias("site_lat"),
    sites_expanded["longitude"].alias("site_lon"),
    amenities_renamed["latitude"].alias("amenity_lat"),
    amenities_renamed["longitude"].alias("amenity_lon"),
    sites_expanded["h3_index"].alias("site_h3"),
    amenities_renamed["amenity_h3"]
).distinct()  # deduplicate — site may reach same amenity via multiple neighbor cells

print(f"H3 candidate pairs: {candidates.count():,}")


# Step 3: Haversine exact distance on candidates 
fact_site_amenity_new = (
    candidates
    .withColumn(
        "distance_meters",
        haversine_meters(
            col("site_lat"), col("site_lon"),
            col("amenity_lat"), col("amenity_lon")
        )
    )
    # Keep only pairs within max search radius
    .filter(col("distance_meters") <= MAX_DISTANCE_M)
    # Classify into relationship type
    .withColumn(
        "relationship_type",
        when(col("distance_meters") <= 100,  lit("within_100m"))
        .when(col("distance_meters") <= 300,  lit("within_300m"))
        .when(col("distance_meters") <= 500,  lit("within_500m"))
        .otherwise(lit("within_1km"))
    )
    .withColumn(
        "h3_same_cell",
        col("site_h3") == col("amenity_h3")
    )
    .withColumn(
        "relationship_id",
        sha2(concat(col("site_id"), lit("_"), col("amenity_id")), 256)
    )
    .select(
        col("relationship_id"),         # PK
        col("site_id"),                 # FK → dim_site
        col("amenity_id"),              # FK → dim_hotel or dim_restaurant
        col("amenity_type"),            # hotel | restaurant
        col("distance_meters"),
        col("relationship_type"),       # within_100m | within_300m | within_500m | within_1km
        col("h3_same_cell"),
        current_date().alias("compute_date"),
        current_date().alias("load_date")
    )
)

print(f"Pairs within {MAX_DISTANCE_M}m: {fact_site_amenity_new.count():,}")

In [0]:
# MERGE into fact_site_amenity 
# Update distance if amenity location changed, insert new pairs
if spark.catalog.tableExists("cre_catalog.silver.fact_site_amenity"):
    dt = DeltaTable.forName(spark, "cre_catalog.silver.fact_site_amenity")
    dt.alias("tgt").merge(
        fact_site_amenity_new.alias("src"),
        "tgt.relationship_id = src.relationship_id"
    ).whenMatchedUpdate(condition="""
        tgt.distance_meters != src.distance_meters
    """, set={
        "distance_meters":    "src.distance_meters",
        "relationship_type":  "src.relationship_type",
        "h3_same_cell":       "src.h3_same_cell",
        "compute_date":       "src.compute_date",
        "load_date":          "src.load_date"
    }).whenNotMatchedInsertAll().execute()
    print("fact_site_amenity merged")
else:
    fact_site_amenity_new.write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("cre_catalog.silver.fact_site_amenity")
    print("fact_site_amenity created")

print(f"   Total rows: {spark.table('cre_catalog.silver.fact_site_amenity').count():,}")